## Cài thư viện, chronium trước khi chạy 

- pip install -r requirement.txt
- playwright install chronium

In [4]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import os

async def crawl_flexible():
    async with async_playwright() as p:
        user_data_dir = os.path.expanduser("~/playwright_ta_profile_interactive")
        
        print("  Đang khởi chạy trình duyệt...")
        context = await p.chromium.launch_persistent_context(
            user_data_dir=user_data_dir,
            headless=False,
            executable_path='/usr/bin/chromium-browser',
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-setuid-sandbox'],
            no_viewport=True
        )
        
        page = context.pages[0]
    
        # Giải CAPTCHA và COPY URL trên thanh địa chỉ DÁN vào tùy theo tỉnh thành hoặc nội dung cần crawl. 
        url = "https://www.tripadvisor.com/Search?q=v%C5%A9ng+t%C3%A0u&ssrc=A&searchNearby=false&searchSessionId=00052ab4363c51de.ssid" 
        
        print(f"  Đang truy cập: {url}")
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        
        # Chờ 5 giây để  giải CAPTCHA nếu hiện ra, hoặc để JS render xong
        print("  Chờ 15 giây để trang ổn định (hoặc chờ bạn giải CAPTCHA)...")
        await page.wait_for_timeout(15000)
        
        html = await page.content()
        soup = BeautifulSoup(html, 'html.parser')
        
        # Debug: In ra tiêu đề trang
        page_title = soup.title.string.strip() if soup.title else "Không có tiêu đề"
        print(f"  Tiêu đề trang hiện tại: '{page_title}'")
        
        # Thử nhiều selector khác nhau
        results_list = soup.find('div', {'data-test-target': 'results-list'})
        
        # Nếu không tìm thấy, thử tìm theo class phổ biến của TripAdvisor
        if not results_list:
            results_list = soup.find('div', class_=lambda c: c and 'listing' in c.lower())
            
        if not results_list:
            print("  Không tìm thấy container chứa danh sách địa điểm.")
            await context.close()
            return None
            
        # Tìm các card địa điểm (thử nhiều attribute có thể có)
        cards = results_list.find_all('div', {'data-test-attribute': 'location-results-card'})
        
        # Fallback: Nếu vẫn không ra, tìm tất cả các thẻ a có target="_blank" nằm trong khu vực results_list
        if not cards:
            print("  Không tìm thấy 'location-results-card', đang thử phương án dự phòng...")
            # Tìm các khối có chứa link và tên
            cards = results_list.find_all('a', target='_blank')
            print(f"  Tìm thấy {len(cards)} liên kết địa điểm (phương án dự phòng).")
            
            data = []
            for card in cards[:20]: # Giới hạn 20 kết quả cho phương án dự phòng
                name = card.get_text(strip=True)
                # Với phương án dự phòng, việc lấy rating/review khó hơn vì cấu trúc khác, ta tạm thời để N/A
                data.append({
                    "Tên địa điểm": name,
                    "Điểm đánh giá": "N/A (Cần điều chỉnh selector)",
                    "Số lượng đánh giá": "N/A (Cần điều chỉnh selector)"
                })
        else:
            print(f"Đã tìm thấy {len(cards)} địa điểm (cấu trúc chuẩn).")
            data = []
            for card in cards:
                name_tag = card.find('a', target='_blank')
                name = name_tag.get_text(strip=True) if name_tag else "N/A"
                
                rating_div = card.find('div', {'data-automation': 'bubbleRatingValue'})
                rating_span = rating_div.find('span') if rating_div else None
                rating = rating_span.get_text(strip=True) if rating_span else "N/A"
                
                review_span = card.find('span', {'data-automation': 'bubbleReviewCount'})
                review_count = review_span.get_text(strip=True) if review_span else "N/A"
                
                data.append({
                    "Tên địa điểm": name,
                    "Điểm đánh giá": rating,
                    "Số lượng đánh giá": review_count
                })
            
        df = pd.DataFrame(data)
        print("\n--- Dữ liệu thu thập được ---")
        display(df)
        
        loop = asyncio.get_running_loop()
        await loop.run_in_executor(None, input, "\nNhấn phím ENTER trên Jupyter Notebook để đóng cửa sổ trình duyệt...")
        
        await context.close()
        return df


# Chạy hàm
df = await crawl_flexible()
df.to_csv("VungTau.csv")

  Đang khởi chạy trình duyệt...
  Đang truy cập: https://www.tripadvisor.com/Search?q=v%C5%A9ng+t%C3%A0u&ssrc=A&searchNearby=false&searchSessionId=00052ab4363c51de.ssid
  Chờ 15 giây để trang ổn định (hoặc chờ bạn giải CAPTCHA)...
  Tiêu đề trang hiện tại: 'Tripadvisor'
Đã tìm thấy 30 địa điểm (cấu trúc chuẩn).

--- Dữ liệu thu thập được ---


,Tên địa điểm,Điểm đánh giá,Số lượng đánh giá
0,Things to doJesus Christ Statue4.34.3 of 5 bub...,4.3,(886 reviews)
1,Things to doVung Tau Lighthouse4.14.1 of 5 bub...,4.1,(251 reviews)
2,"Things to doVung Tau Essential: Beach, Christ ...",4.9,(469 reviews)
3,Things to doVung Tau Ferry Terminal3.73.7 of 5...,3.7,(198 reviews)
4,Things to doBest Tour In Vung Tau4.84.8 of 5 b...,4.8,(116 reviews)
5,Things to doBãi Biển Vũng Tàu3.93.9 of 5 bubbl...,3.9,(166 reviews)
6,Things to doThe Robert Taylor Museum Of Worldw...,4.8,(247 reviews)
7,Things to doPrincess Spa Vung Tau4.74.7 of 5 b...,4.7,(71 reviews)
8,Things to doThe Whale Temple Vung Tau3.43.4 of...,3.4,(73 reviews)
9,Things to doGreenlinesDP Fast Ferry4.24.2 of 5...,4.2,(180 reviews)
